# $H_2O$ Molecule: Charge Density
---

<font size="5"><b>Goal:</b>
In this example, we compute the electronic charge density of
the water molecule at its equilibrium geometry.
</font>

&nbsp;

&nbsp;

## In Development!!
-------------------

- need to figure out how to get the unit volume from PySCF.

## **Run the code block below to set up the example**

In [ ]:
# Run me (shift+enter or click the play button) to setup the tutorial!
from pathlib import Path

import h5py as h5
import numpy as np
from pyscf import gto,scf,mcscf

from afqmctools.utils.pyscf_utils import load_from_pyscf_chk_mol
from afqmctools.hamiltonian.mol import write_hamil_mol
from afqmctools.wavefunction.mol import write_cas_wfn
from afqmctools.inputs.from_hdf import write_json
from afqmctools.hamiltonian.io import write_to_hdf5

from stats.scalar_dat import analyze_scalar_data

from tutorial_utils import run_afqmc, get_scratch_dir

# For you TODO: set a scratch directory for the files that will be generated
home = Path.home()
scratch_dir = get_scratch_dir("example_h2o_density",home / ".scratch")

## The Water Molecule
---------------------

In [ ]:
"""
minimial example for understanding how PySCF computes the charge density and dipole moment.

The sum of rho does not equal
the total number of electrons...

"""

from pathlib import Path

import numpy as np
from pyscf import gto,scf

from pyscf.scf.hf import dip_moment
from pyscf.tools import cubegen

scratch = scratch_dir
chkfile = scratch / "uhf.chk"

"""
Equilibrium geometry of $H_2O$:

a0 = 0.9572
theta = 104.52 degrees
"""

a0 = 0.9572
theta = 104.52
ay = a0*np.cos(np.radians(theta/2))
ax = a0*np.sin(np.radians(theta/2))

atoms = f"""
O 0.000 0.000 0.000
H {ax} {-ay} 0.0
H {-ax} {-ay} 0.0
"""

mol = gto.M(
    atom = atoms,
    spin = 0,
    basis = 'cc-pvdz',
    verbose = 4
)

mf = scf.UHF(mol).newton()
mf.chkfile = chkfile
mf.kernel()

rdm = mf.make_rdm1()

nx = ny = nz = 3

rho = cubegen.density(mol,'h2o_den.cube', rdm, nx=nx, ny=ny, nz=nz)


mycube = cubegen.Cube(mol, nx=nx, ny=ny, nz=nz)
mycube.read('h2o_den.cube')
volume_element = mycube.get_volume_element() # NOTE: this is the volume in fractional coords!!! Need actual volume element!!

ngrids = rho.shape[0] * rho.shape[1] * rho.shape[2]
print(f"Integrated charge = {np.sum(rho)*volume_element}")
print(f"grid size = {nx} x {ny} x {nz}")
print(f"volume element = {volume_element}")


#print(f"Integrated charge = {np.sum(rho)}")
#print(f"Number of grid points = {ngrids}")

#print(f"Charge per volume element = {np.sum(rho)*volume_element}")



System: uname_result(system='Linux', node='ccqlin053.flatironinstitute.org', release='4.18.0-553.63.1.el8_10.x86_64', version='#1 SMP Thu Jul 24 11:45:38 UTC 2025', machine='x86_64')  Threads 32
Python 3.11.7 (main, May  1 2024, 20:55:17) [GCC 11.4.0]
numpy 2.2.6  scipy 1.16.1  h5py 3.14.0
Date: Fri Aug 15 15:58:47 2025
PySCF version 2.10.0
PySCF path  /home/beskridge/local_software/SAFIRE/build/venv_mods2.3/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 H      0.756950327264  -0.585882276618   0.0000000

In [ ]:
dir(rho)
#print(rdm.shape)
#print(rho.shape)

['T',
 '__abs__',
 '__add__',
 '__and__',
 '__array__',
 '__array_finalize__',
 '__array_function__',
 '__array_interface__',
 '__array_namespace__',
 '__array_priority__',
 '__array_struct__',
 '__array_ufunc__',
 '__array_wrap__',
 '__bool__',
 '__class__',
 '__class_getitem__',
 '__complex__',
 '__contains__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dir__',
 '__divmod__',
 '__dlpack__',
 '__dlpack_device__',
 '__doc__',
 '__eq__',
 '__float__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__ifloordiv__',
 '__ilshift__',
 '__imatmul__',
 '__imod__',
 '__imul__',
 '__index__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__irshift__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__lshift__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '_

# analyze with afqmctools

In [ ]:
from pathlib import Path

from pyscf.tools import cubegen
import numpy as np
import h5py as h5
import matplotlib.pyplot as plt
from pyscf import gto

from pyscf.scf.hf import dip_moment

from afqmctools.utils.pyscf_utils import load_from_pyscf_chk_mol

scratch = Path("./scratch")
scratch.mkdir(exist_ok=True)

reference = Path("./ref_results")
reference.mkdir(exist_ok=True)

afqmc_run_dir = Path("./run_afqmc")
afqmc_run_dir.mkdir(exist_ok=True)


uhf_results = reference / "uhf_rdm.h5"
dft_results = reference / "dft_pbe.h5"
cc_results = reference / "cc.h5"

scf_data_basis = load_from_pyscf_chk_mol(scratch / "rhf.chk")

mo_coeff = scf_data_basis["mo_coeff"]

a0 = 0.9572
theta = 104.52
ay = a0*np.cos(np.radians(theta))
ax = a0*np.sin(np.radians(theta))

atoms = f"""
O 0.000 0.000 0.000
H {ax} {ay} 0.0
H {-ax} {ay} 0.0
"""

x_min = -1.2
x_max = 1.2
y_min = -1.2
y_max = 1.2
x_points = 200
y_points = 200


def rdm_mo2ao(mol,rdm_mo,C):
    r"""
    Convert RDM from MO basis to AO basis.

    .. math::

        \gamma_{\mu\nu} = S_{\mu\mu'} C_{\mu' i}  \gamma_{ij} C^*_{j \nu'} S_{\nu' \nu}

    """
    S = mol.intor('int1e_ovlp')
    C = np.array(C)
    X = S @ C
    rdm_ao = X @ rdm_mo @ X.conj().T
    return rdm_ao

def plot_xy_density(spin_summed_rho,label=""):
    # make 2d density plot in the x-y plane at z=0
    x = np.linspace(x_min, x_max, x_points)
    y = np.linspace(y_min, y_max, y_points)

    if spin_summed_rho.shape[0] == 2:
        spin_summed_rho = np.sum(spin_summed_rho, axis=0)

    # Create coordinate matrices
    X, Y = np.meshgrid(x, y)

    #import pdb; pdb.set_trace()

    # Example function to evaluate on the grid (e.g., Gaussian function)
    rho = cubegen.density(mol, f"density{"_"+label if label != "" else ""}.cube", spin_summed_rho, nx=200, ny=200, nz=2)

    # Plot the 2D density plot
    plt.figure(figsize=(8, 6))
    plt.matshow(
        100*rho[:,:,0],
        cmap="hot",#"bone",#"Purples" #"viridis",
        interpolation="bicubic"
    )
    plt.colorbar(label="Density")

    # Set tick positions and labels
    tick_positions_x = np.linspace(0, x_points - 1, 5, dtype=int)
    tick_labels_x = np.round(np.linspace(x_min, x_max, 5), 2)
    tick_positions_y = np.linspace(0, y_points - 1, 5, dtype=int)
    tick_labels_y = np.round(np.linspace(y_min, y_max, 5), 2)

    plt.gca().set_xticks(tick_positions_x)
    plt.gca().set_xticklabels(tick_labels_x)
    plt.gca().set_yticks(tick_positions_y)
    plt.gca().set_yticklabels(tick_labels_y)

    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title("Charge Density of $H_2O$ in X-Y plane")
    plt.show()


print("Input Molecule")
print(atoms)

mol = gto.M(
    atom = atoms,
    spin = 0,
    basis = 'cc-pvdz',
    verbose = 4
)

with h5.File(uhf_results,"r") as f:
    rdm_uhf = f["rdm_uhf"][:]

print("UHF Density")
rho = cubegen.density(mol, "UHF_density.cube", rdm_uhf, nx=200, ny=200, nz=1)

plot_xy_density(rdm_uhf,label="UHF")

xcut_idx = 12 #np.argmax(np.sum(rho_uhf, axis=(1,2)))
print(f"Integrated charge = {np.sum(rho)}")

if False:
    plt.matshow(
        100*rho[xcut_idx],
        cmap="bone",#"Purples" #"viridis",
        interpolation="bicubic"
    )
    plt.show()

mu_vec =  dip_moment(mol,rdm_uhf,unit="Ha")
print(f"mu vector = {mu_vec}")
print(f"Dipole moment = {np.sqrt(np.sum(np.power(mu_vec,2)))}")


with h5.File(dft_results,"r") as f:
    rdm_dft = f["rdm"][:]

print("DFT-PBE Density")
rho = cubegen.density(mol, "DFT_density.cube", rdm_dft, nx=200, ny=200, nz=1)


plot_xy_density(rdm_dft,"DFT")

xcut_idx = 12 #np.argmax(np.sum(rho_uhf, axis=(1,2)))
print(f"Integrated charge = {np.sum(rho)}")

if False:
    plt.matshow(
        100*rho[xcut_idx],
        cmap="bone",#"Purples" #"viridis",
        interpolation="bicubic"
    )
    plt.show()

mu_vec =  dip_moment(mol,rdm_dft,unit="Ha")
print(f"mu vector = {mu_vec}")
print(f"Dipole moment = {np.sqrt(np.sum(np.power(mu_vec,2)))}")

with h5.File(cc_results,"r") as f:
    rdm_cc = f["rdm_cc"][:]

print("CCSD Density")
#rho = cubegen.density(mol, "CC_density.cube", rdm_cc, nx=200, ny=200, nz=1)
#plot_xy_density(rho,"CCSD")

xcut_idx = 12 #np.argmax(np.sum(rho_uhf, axis=(1,2)))
print(f"Integrated charge = {np.sum(rho)}")

if False:
    plt.matshow(
        100*rho[xcut_idx],
        cmap="bone",#"Purples" #"viridis",
        interpolation="bicubic"
    )
    plt.show()

mu_vec =  dip_moment(mol,rdm_cc,unit="Ha")
print(f"mu vector = {mu_vec}")
print(f"Dipole moment = {np.sqrt(np.sum(np.power(mu_vec,2)))}")

from afqmctools.analysis.rdm import average_afqmc_rdm

rho_avg, delta_rho = average_afqmc_rdm(
    rdm_file=afqmc_run_dir/"qmc.s000.stat.h5"
)

naverages = rho_avg.shape[0]
nspins = rho_avg.shape[1]
nmo = rho_avg.shape[2]


print("AFQMC Density")
for average in range(naverages):

    spin_summed_rho = np.sum(rho_avg[average],axis=0)

    # TODO: convert to ao basis!!!
    rdm_ao = rdm_mo2ao(mol,spin_summed_rho,mo_coeff)

    print(f"Average {average}")
    rho = cubegen.density(mol, f"AFQMC_density_avg{average}.cube", rdm_ao, nx=21, ny=100, nz=100)
    xcut_idx = 12
    print(f"Integrated charge = {np.sum(rho[average])}")
    if False:
        plt.matshow(
            100*rho[xcut_idx],
            cmap="bone",#"Purples" #"viridis",
            interpolation="bicubic"
        )
        plt.show()

    mu_vec =  dip_moment(mol,rdm_ao,unit="Ha")

    print(f"mu vector = {mu_vec}")
    print(f"Dipole moment = {np.sqrt(np.sum(np.power(mu_vec,2)))}")

    # make 2d density plot in the x-y plane at z=0
    x = np.linspace(x_min, x_max, x_points)
    y = np.linspace(y_min, y_max, y_points)

    # Create coordinate matrices
    X, Y = np.meshgrid(x, y)

    # Example function to evaluate on the grid (e.g., Gaussian function)
    #rho = cubegen.density(mol, "AFQMC_density.cube", rdm_ao, nx=200, ny=200, nz=1)

    # Plot the 2D density plot
    #plt.figure(figsize=(8, 6))
    #plt.matshow(
    #    100*rho[xcut_idx],
    #    cmap="bone",#"Purples" #"viridis",
    #    interpolation="bicubic"
    #)
    #plt.colorbar(label="Density")
    #plt.xlabel("X")
    #plt.ylabel("Y")
    #plt.title("Charge Density of $H_2O$ in X-Y plane")
    #plt.show()

print("Done")
